# 02 Bronze - Telecommunications

**Audience:** participants learning the AIDP medallion pattern with PySpark.

**Prerequisites:** use the lab's shared compute and run the previous notebook first.

**Learning goal:** Preserves source values and lineage while converting each dataset to Delta.

## Outline

1. Inspect the participant-scoped inputs.
2. Transform and persist this medallion layer.
3. Register external tables when this layer owns them.
4. Verify the row counts printed by the final statements.


In [ ]:
import re
# oidlUtils is injected by AIDP Workbench; no import is required.

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")
catalog_name = required_parameter("catalog_name")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != 'telecommunications':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")
if catalog_name != f"{participant_key}_aidp_lab":
    raise ValueError("Invalid participant catalog")
spark.conf.set("spark.aidp.lineage.enabled", "true")

def table(layer, logical_name):
    prefix = f"{lab_id}_"
    physical_name = logical_name if logical_name.startswith(prefix) else prefix + logical_name
    return f"{catalog_name}.oci_{layer}.{participant_key}_{physical_name}"

from pyspark.sql import functions as F

specs = {'plans': {'filename': 'plans.csv', 'primary_key': ['plan_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'plan_id', 'type': 'STRING', 'required': True}, {'name': 'plan_type', 'type': 'STRING', 'required': True}, {'name': 'monthly_fee', 'type': 'DOUBLE', 'required': True}, {'name': 'included_data_mb', 'type': 'BIGINT', 'required': True}, {'name': 'included_voice_minutes', 'type': 'BIGINT', 'required': True}, {'name': 'overage_rate', 'type': 'DOUBLE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'network_sites': {'filename': 'network_sites.csv', 'primary_key': ['site_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'site_id', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'technology', 'type': 'STRING', 'required': True}, {'name': 'capacity_mb_day', 'type': 'BIGINT', 'required': True}, {'name': 'commissioned_date', 'type': 'DATE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'subscribers': {'filename': 'subscribers.csv', 'primary_key': ['subscriber_id'], 'foreign_keys': [['plan_id', 'plans', 'plan_id'], ['home_site_id', 'network_sites', 'site_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'subscriber_id', 'type': 'STRING', 'required': True}, {'name': 'plan_id', 'type': 'STRING', 'required': True}, {'name': 'home_site_id', 'type': 'STRING', 'required': True}, {'name': 'segment', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'activation_date', 'type': 'DATE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'usage_events': {'filename': 'usage_events.csv', 'primary_key': ['event_id'], 'foreign_keys': [['subscriber_id', 'subscribers', 'subscriber_id'], ['site_id', 'network_sites', 'site_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'event_id', 'type': 'STRING', 'required': True}, {'name': 'subscriber_id', 'type': 'STRING', 'required': True}, {'name': 'site_id', 'type': 'STRING', 'required': True}, {'name': 'event_time', 'type': 'TIMESTAMP', 'required': True}, {'name': 'usage_type', 'type': 'STRING', 'required': True}, {'name': 'usage_value', 'type': 'DOUBLE', 'required': True}, {'name': 'usage_unit', 'type': 'STRING', 'required': True}, {'name': 'charge_amount', 'type': 'DOUBLE', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}}
sources = {"network_sites": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/telecommunications/network_sites/", "plans": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/telecommunications/plans/", "subscribers": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/telecommunications/subscribers/", "usage_events": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/telecommunications/usage_events/"}
destinations = {"network_sites": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/telecommunications/network_sites/", "plans": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/telecommunications/plans/", "subscribers": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/telecommunications/subscribers/", "usage_events": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/telecommunications/usage_events/"}
landing_tables = {"network_sites": f"{participant_key}_telecommunications_network_sites", "plans": f"{participant_key}_telecommunications_plans", "subscribers": f"{participant_key}_telecommunications_subscribers", "usage_events": f"{participant_key}_telecommunications_usage_events"}

for dataset, spec in specs.items():
    frame = (spark.table(table("landing", dataset))
        .withColumn("_source_file", F.input_file_name())
        .withColumn("_ingested_at", F.current_timestamp()))
    landing_count = frame.count()
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table("bronze", dataset))
    bronze_count = spark.table(table("bronze", dataset)).count()
    assert bronze_count == landing_count, f"Bronze count mismatch for {dataset}"
    print(f"Bronze {dataset}: {bronze_count} rows")


## Expected result

Four Landing CSV tables and four Bronze Delta tables are registered.

**Exercise:** rerun this notebook and confirm that counts do not increase. All writes use
participant-exclusive paths and overwrite mode, so a second run is idempotent.

**Common pitfall:** do not replace the participant paths with shared locations. That would mix
different students' data. As an extension, query the registered tables with `spark.sql`.
